# AnchorDECODE: SPOTS mouse spleen

This notebook runs the same example as `examples/spots.py`: estimate five cell-type proportions from 2,653 tissue spots and 21 antibody features in spleen replicate 1. It uses raw protein counts, spatial coordinates, and literature-supported markers.

The recommended environment installs PyTorch 2.11.0 + CUDA 12.8 for NVIDIA GPUs. Install it from the repository root with `conda env create -f environment.yml`, then activate it with `conda activate anchordecode`. Launch `jupyter lab` from the repository root and select this environment's Python kernel.

See [the complete tutorial](../docs/tutorial.md) for parameter definitions and use with your own data.


## 1. Download the public inputs

Data are not bundled. Download the following files from [GEO GSE198353](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE198353) and place them in `data/SPOTS/` under the repository root:

- [GSE198353_spleen_rep_1_filtered_feature_bc_matrix.h5](https://ftp.ncbi.nlm.nih.gov/geo/series/GSE198nnn/GSE198353/suppl/GSE198353_spleen_rep_1_filtered_feature_bc_matrix.h5)
- [GSE198353_spleen_replicate_1_spatial.tar.gz](https://ftp.ncbi.nlm.nih.gov/geo/series/GSE198nnn/GSE198353/suppl/GSE198353_spleen_replicate_1_spatial.tar.gz)

Keep the spatial archive compressed. Source study: *Integration of whole transcriptome spatial profiling with protein markers*, [DOI: 10.1038/s41587-022-01536-3](https://doi.org/10.1038/s41587-022-01536-3), [PMID: 36593397](https://pubmed.ncbi.nlm.nih.gov/36593397/).


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import display, Image

# Locate the repository whether the kernel starts at its root or in tutorials/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "model/AnchorDECODE.py").is_file()
             and (p / "examples/spots.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter from this repository or its tutorials directory")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from model import AnchorDECODE
from examples.spots import load_spots, plot_proportions

DATA_DIR = ROOT / "data/SPOTS"
OUTPUT_DIR = ROOT / "results/SPOTS"
DEVICE = "cuda"  # Use "cpu" for a CPU-only installation.


## 2. Read the protein counts and spatial coordinates

The reader selects `Antibody Capture` features from the deposited HDF5 matrix, preserving their raw counts and order. Coordinates are matched by barcode; `x` and `y` are full-resolution image columns and rows. No external normalization or log transformation is applied.


In [ ]:
expression, coords = load_spots(DATA_DIR)
markers = json.loads((ROOT / "examples/spots_markers.json").read_text())

print(f"{expression.shape[0]:,} spots × {expression.shape[1]} proteins")
assert expression.index.equals(coords.index)
display(expression.head())
display(pd.DataFrame([
    {"Cell type": cell_type, "Markers": ", ".join(proteins)}
    for cell_type, proteins in markers.items()
]))


## 3. Train AnchorDECODE

The experimental defaults use 5 spatial folds, seeds 42/43/44, and 180 epochs per network (15 networks in total). `epochs=2` can be used for a quick installation check, but does not reproduce the experimental training setting.

Each section is fitted independently. Spatial folds govern neural training; marker calibration uses the full target section. No ground-truth proportions or external reference profiles are used in fitting.


In [ ]:
model = AnchorDECODE(device=DEVICE)
display(pd.DataFrame(model.config.items(), columns=["Parameter", "Value"]))
result = model.train(expression, coords, markers)
prediction = result["prediction"]

assert prediction.shape == (len(expression), len(markers))
assert prediction.index.equals(expression.index)
assert np.isfinite(prediction.to_numpy()).all()
assert (prediction.to_numpy() >= 0).all()
np.testing.assert_allclose(prediction.sum(axis=1), 1.0, rtol=0, atol=1e-10)
display(prediction.head())


## 4. Inspect marker coverage and export

Each row of the final table is a spot and each column is a cell type. Estimates sum to one across the five modeled populations. These populations need not include every cell type present in the tissue.

The export includes final and intermediate proportions, marker diagnostics, model settings, spatial folds, training losses, and a fitted checkpoint.


In [ ]:
display(model.seed_details_["coverage"])
display(model.seed_details_["reliability"])
model.export(OUTPUT_DIR)
coords.to_csv(OUTPUT_DIR / "coordinates.csv")
print("Saved outputs to:", OUTPUT_DIR)


## 5. Plot spatial proportions

Maps use the same 0–1 color scale and preserve image orientation. The figure visualizes the estimates; it is not a comparison against measured cell-type proportions.


In [ ]:
plot_proportions(prediction, coords, OUTPUT_DIR / "proportions.png")
display(Image(filename=str(OUTPUT_DIR / "proportions.png")))


## 6. Reload the fitted model

`predict()` replays the fitted section's out-of-fold predictions. It requires the identical expression values, numeric dtype, identifiers, and ordering used during training. Train a new instance on a new section.


In [ ]:
restored = AnchorDECODE.load(OUTPUT_DIR / "model.pt", device=DEVICE)
replayed = restored.predict(expression)
np.testing.assert_allclose(replayed.to_numpy(), prediction.to_numpy(),
                           rtol=1e-5, atol=1e-7)
print("Checkpoint reloaded; predictions agree.")


## Continue with your own data

Prepare a raw spot-by-protein DataFrame, an aligned two-column coordinate DataFrame, and a marker dictionary. Retain the full measured protein panel, including non-marker features. Fit each tissue section independently with `AnchorDECODE().train(expression, coords, markers)`.

See [Use your own spatial proteomics data](../docs/tutorial.md#use-your-own-spatial-proteomics-data) for details. Marker references and parameter descriptions are also provided in that guide.
